In [9]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import mysql.connector

In [15]:
def generate_dagevolve_erd(connection, database="dagevolve"):
    """
    Auto-generate a Mermaid ERD for the Dagevolve schema.
    Reads tables, columns, PKs, and FKs from MySQL.
    """

    cursor = connection.cursor(dictionary=True)

    # Get all tables in the Dagevolve schema
    cursor.execute(f"""
        SELECT TABLE_NAME 
        FROM INFORMATION_SCHEMA.TABLES
        WHERE TABLE_SCHEMA = '{database}';
    """)
    tables = [row["TABLE_NAME"] for row in cursor.fetchall()]

    erd = ["erDiagram"]

    for table in tables:
        # Get columns
        cursor.execute(f"""
            SELECT COLUMN_NAME, COLUMN_TYPE, COLUMN_KEY
            FROM INFORMATION_SCHEMA.COLUMNS
            WHERE TABLE_SCHEMA = '{database}'
              AND TABLE_NAME = '{table}';
        """)
        cols = cursor.fetchall()

        erd.append(f"    {table} {{")
        for col in cols:
            col_type = col["COLUMN_TYPE"]
            col_name = col["COLUMN_NAME"]
            erd.append(f"        {col_type} {col_name}")
        erd.append("    }")

        # Get foreign keys
        cursor.execute(f"""
            SELECT 
                COLUMN_NAME,
                REFERENCED_TABLE_NAME,
                REFERENCED_COLUMN_NAME
            FROM INFORMATION_SCHEMA.KEY_COLUMN_USAGE
            WHERE TABLE_SCHEMA = '{database}'
              AND TABLE_NAME = '{table}'
              AND REFERENCED_TABLE_NAME IS NOT NULL;
        """)
        fks = cursor.fetchall()

        for fk in fks:
            erd.append(
    f"    {table} }}o--|| {fk['REFERENCED_TABLE_NAME']} : {fk['COLUMN_NAME']} -> {fk['REFERENCED_COLUMN_NAME']}"
)


    cursor.close()
    return "\n".join(erd)


In [19]:
import mysql.connector

conn = mysql.connector.connect(
    host="127.0.0.1",
    user="root",
    password="Catalyst@312",
    database="dagevolve"
)

erd_code = generate_dagevolve_erd(conn)
print(erd_code)

erDiagram
    dagevolve_full {
        varchar(50) Console
        int Console Disco Year
        int Console Release Year
        text Console_Company
        double Consoles sold (million)
        decimal(10,2) Europe_Sales
        text Game Title
        varchar(255) Game_Company
        text Genre
        decimal(10,2) Global_Sales
        decimal(10,2) Japan_Sales
        decimal(10,2) North America_Sales
        text Type
        int Year
    }
    gameconsoles {
        text Company
        text Console
        int Console Disco Year
        int Console Release Year
        double Consoles sold (million)
        text Remarks
        text Type
    }
    gamesales {
        varchar(255) Company
        varchar(50) Console
        decimal(5,2) Europe_Sales
        text Game Title
        text Genre
        double Global_Sales
        decimal(5,2) Japan_Sales
        decimal(5,2) North America_Sales
        int Year
    }
